# Recursive subagents: Context management and scope

In JAZ, an `invoke` agent can recursively call `invoke` by default. In other words, subagents, subsubagents,
etc. are available by default.
(This is a [natural consequence of the functional `invoke` framework](https://arxiv.org/pdf/2609.26891#page=3).)

Subagents are traditionally used for a particular form of context management: decomposition into subtasks.
However, JAZ enables more expressive forms of context management with subagents, allowing `invoke` to
[perform long-horizon tasks requiring long-range recall and self-improve across a sequence of tasks](https://arxiv.org/pdf/2609.26891#page=5). This is because [everything the LLM sees, all inputs to `invoke` as well as
the REPL history (`__history__`), are available as variables in the REPL](https://arxiv.org/pdf/2609.26891#page=3),
allowing the LLM to efficiently build a context
based on its own context to give to another agent (either a subagent or its parent agent).

The presence of subagents raises the natural scoping question about [inputs](01-inputs.ipynb) to top-level `invoke`:
sometimes we may want an input to also propagate automatically to subagents, e.g., a system-level instruction or a tool
that you want to make available to all (sub)agents.

This tutorial demonstrates how JAZ `invoke` can use subagents for expressive forms of context management,
and introduces JAZ's scoping mechanism for controlling whether an input
to `invoke` automatically makes it available to all its recursive subagents.

## Setup

To set up this notebook, install JAZ, provide an API key, and configure a model:

In [ ]:
%pip install -q jaz-lang pandas

In [1]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [ ]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4'))

We'll also define a helper hook that prints each agent's prompt and trajectory, labeled with which
agent it is: `root` for the top-level agent, `root.1` for its first subagent, `root.1.2` for that
subagent's second subagent, and so on.
*You do not need to understand it now — hooks are introduced in [tutorial 3](03-hooks-events-and-effects.ipynb)*.

In [ ]:
import sys
import threading
from jaz.hooks import Hook


class Logger(Hook):
    """Prints each agent's prompt and trajectory, labeled with which agent it is.

    Only the first `subagents` subagents of each agent are logged (all of them if `None`).
    With `system_prompt=True`, the end of each agent's system prompt is printed too.
    """

    def __init__(self, subagents: int | None = None, system_prompt: bool = False):
        self.subagents = subagents
        self.system_prompt = system_prompt
        # Inside an agent's REPL, `sys.stdout` is captured as that agent's REPL output, so we
        # print to the `sys.stdout` of now. Otherwise a subagent's log would end up in its parent's context.
        self.out = sys.stdout
        # Agents can run in parallel threads, so we identify them by their `invoke_id`.
        self.lock = threading.Lock()
        self.agents = {}         # invoke_id -> (label, logged?)
        self.num_subagents = {}  # invoke_id -> number of subagents it has launched so far

    def on_invoke_enter(self, event):
        with self.lock:
            parent_id = event.parent_invoke_id
            if parent_id not in self.agents:
                label, logged = "root", True
            else:
                parent, parent_logged = self.agents[parent_id]
                n = self.num_subagents[parent_id] = self.num_subagents.get(parent_id, 0) + 1
                label = f"{parent}.{n}"
                logged = parent_logged and (self.subagents is None or n <= self.subagents)
            self.agents[event.invoke_id] = (label, logged)

    def log(self, event, title, text):
        label, logged = self.agents[event.invoke_id]
        if logged:
            with self.lock:
                print(f"--- [{label}] {title} BEGINS ---\n{text}\n--- [{label}] {title} ENDS ---", file=self.out)

    def on_llm_query_enter(self, event):
        if event.iteration == 0:
            if self.system_prompt:
                # The part after the description of `__history__`, which is where scoped inputs are shown.
                self.log(event, "SYSTEM PROMPT (END)", event.messages[0]["content"].split("</__history__>")[-1].strip())
            self.log(event, "USER PROMPT", event.messages[-1]["content"])
        else:
            self.log(event, "REPL OUTPUT", event.messages[-1]["content"])

    def on_llm_query_complete(self, event):
        self.log(event, "LLM RESPONSE", event.response.content)

Subagents can take a minute or more in total, which is longer than the REPL's default limit of 30
seconds per code execution. We lift that limit for this tutorial, and allow the agents to import
`pandas`, so that they can return DataFrames, and `concurrent`, so that they can run subagents in
parallel threads (*configuration is the subject of [tutorial 4](04-configuration.ipynb)*):

In [ ]:
from jaz.repl import PythonREPL

jaz.configure(repl=PythonREPL(allowed_imports=["pandas", "concurrent"], exec_timeout=None))

## The running example: transcribing field notes

All examples in this tutorial work on the same task. The
[Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) dataset from tutorial 1 was
collected in the field, so let's go back a step: for each of its 344 penguins, we have a note
written in the field, and the agent's job is to turn the notes into a table. The notes are
phrased in many different ways and use different units (mm, cm, inches, g, kg, lb), and some
give the sex only as "she", "hen" or "♂", so a parser won't do. The agent has to read every note.

In [ ]:
from pathlib import Path
import pandas as pd

# The notes are in this repository, next to the notebook; on Colab, we fetch them from GitHub.
NOTES = "data/penguin_field_notes.csv"
if not Path(NOTES).exists():
    NOTES = "https://raw.githubusercontent.com/jaz-lang/cookbook/main/notebooks/" + NOTES
notes = pd.read_csv(NOTES)

# Print every note in full, instead of cutting it off after 50 characters.
pd.set_option("display.max_colwidth", None)

notes

Note how pandas abridges the DataFrame to its first and last five rows (it does so for DataFrames
longer than 60 rows). Since an `invoke` input is
[rendered with its string representation](01-inputs.ipynb), this is also all that an agent given
`notes` sees in its prompt. The other 334 notes are only in its REPL.

## Subagents for subtask decomposition

Recall that the agent's REPL already contains `invoke`, with nothing to import or pass in. Calling it
from the REPL launches a *subagent* (a "sub-invoke"), which is just another `invoke` agent loop, and
returns whatever the subagent returns.

A single agent could print all 344 notes in its REPL and transcribe them, but that fills its context
with 344 notes and 344 rows, and it has to keep track of all of them at once. Instead, we use
subagents to process the DataFrame in chunks,
[RLM-style](https://arxiv.org/abs/2512.24601): each subagent gets a chunk of notes, short enough to be shown in full in its
prompt, and transcribes them, and the top-level agent puts their rows together.

All we add to the task is that the agent should split the notes into small chunks, process each
chunk with a subagent, and run the subagents in parallel; the agent decides how to split the work
and how to run the subagents in parallel (in our runs, it used a thread pool). The
`RecursionLimit(max_depth=2)` hook allows subagents but not subsubagents, so that a subagent can't
hand its chunk on to yet another agent (*hooks are introduced in [tutorial 3](03-hooks-events-and-effects.ipynb)*).
We log the top-level agent (`root`) and its first subagent (`root.1`). This takes about a minute.

In [ ]:
from jaz import invoke
from jaz.hooks import RecursionLimit

transcribe = (
    "Turn `notes`, a DataFrame of field notes on one penguin each, into a table: return a DataFrame "
    "with one row per note, in order, with the columns species (Adelie, Chinstrap or Gentoo), island "
    "(Biscoe, Dream or Torgersen), bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g "
    "(floats, in mm and g) and sex (MALE or FEMALE). Use None for anything not recorded. Make sure "
    "to actually read the notes and transcribe them manually; do NOT use a programmatic parser as "
    "it is brittle."
)
use_subagents = (
    " Split the notes up into multiple small chunks and process each chunk with a subagent. Run the "
    "subagents in parallel."
)

with RecursionLimit(max_depth=2), Logger(subagents=1):
    table = invoke(task=transcribe + use_subagents, notes=notes)

table

In the log, `root` splits `notes` into chunks and calls `invoke` on each one, writing a task for
the subagents in its own words. `root.1`'s prompt shows its whole chunk of notes.

Subtask decomposition is the traditional use of subagents. But since
[everything an agent sees is a variable in its REPL](https://arxiv.org/pdf/2609.26891#page=3), including its own history,
JAZ subagents can do more: a subagent can efficiently hand its context back to its caller, and an
agent can efficiently build a subagent's context from its own context.

## Passing context up: returning `__history__`

Every agent's REPL has a variable `__history__`, the agent's own record of its turns so far. It's a
list with one entry per turn, and each entry has three fields:
- `llm_response`: the code the LLM wrote, comments included;
- `repl_output`: what that code printed;
- `repl_exception`: the exception it raised, if any.

Since `__history__` is an ordinary value, a subagent can simply `return` it, and its caller can
read it. In the next cell, we ask the top-level agent to have each subagent return its
`__history__` along with its part of the table, so that the top-level agent can check how the
subagent did its work: from the table alone, it can't tell whether a subagent read the notes or
ran a parser over them, but from the code the subagent wrote, it can. We log only the top-level
agent this time.

In [ ]:
with RecursionLimit(max_depth=2), Logger(subagents=0):
    table_with_histories = invoke(
        task=transcribe + use_subagents + " Ask each subagent to also return a reference to its "
        "`__history__` so that you can check the subagent's work, e.g., that it transcribed the notes "
        "by hand rather than with programmatic parsing. Make sure to actually read the code each "
        "subagent wrote and check it manually; do NOT use programmatic checks as they are brittle.",
        notes=notes,
    )

In the log, after the subagents return, `root` typically prints the code from their histories
(their `llm_response`s) and reads it in its next turn before putting the table together. The
details vary from run to run: it may, for example, also compare some rows against the notes.

Two things to note. First, the subagents returned their `__history__` *by reference*. A subagent
returns in its last turn, before that turn has been added to its `__history__`; JAZ appends it
afterwards, to the same list. So the history its caller receives is complete, including the turn
that returned it, which is usually the one that contains the transcription. A copy, such as
`list(__history__)`, would have missed that turn. Second, the subagents didn't have to *write out* a
report of their work, which costs output tokens and loses detail. They returned their own record of
their work.

## Passing context down: `__history__[i].repl_output`

The same works in the other direction: an agent can pass parts of its own context to a subagent by
reference.

It turns out that the weights in pounds need more than a unit conversion. They were taken by a
volunteer whose spring scale reads 0.2 lb too high, which is recorded only in the field team's
notebook. No amount of reading the notes reveals it. The notebook is available as a tool,
`view_notebook()`, as in the [overview](00-overview.ipynb), except that this version prints the
notebook instead of returning it:

In [ ]:
def view_notebook() -> None:
    """The field team's notebook from the season the measurements were taken, which records
    known problems with them. Read it before cleaning the data."""
    print(
        "FIELD NOTEBOOK\n"
        "- Torgersen: rough landing; lost a caliper and switched to the spare (checked against the reference: OK).\n"
        "- Biscoe: rain most of the week. Some notes got wet and were recopied in camp.\n"
        "- Dream: back at the station we found that the visiting volunteer's spring scale had not been "
        "tared properly. Every weight they recorded in lb, on any island, reads 0.2 lb too high. Bill and "
        "flipper measurements are unaffected.\n"
        "- All notes were transcribed into the spreadsheet by hand at the station."
    )


view_notebook()

We give `view_notebook` to the top-level agent. Like `help()`, it *prints* its information instead
of returning it, so after calling it, the notebook exists only in the agent's context. The agent
could copy it into a string by retyping it, but that costs output tokens, and an LLM copying text
by hand can make mistakes. Instead, it can pass the REPL output of the turn that called
`view_notebook()` straight to every subagent: `__history__[0].repl_output`.

In [ ]:
with RecursionLimit(max_depth=2), Logger(subagents=1):
    table_with_notebook = invoke(
        task=transcribe + use_subagents + " Call `view_notebook()` first, and pass "
        "`__history__[0].repl_output` to each subagent as an input.",
        notes=notes,
        view_notebook=view_notebook,
    )

In the log, `root` prints the notebook in its first turn, and `root.1`'s prompt contains it as an
input, word for word, without `root` having written it out. Here are the weights given in pounds,
as transcribed before and now:

In [ ]:
in_lb = notes.note.str.contains(r"\d lb")
pd.DataFrame({
    "note": notes.note[in_lb],
    "body_mass_g before": table.body_mass_g[in_lb],
    "body_mass_g with the notebook": table_with_notebook.body_mass_g[in_lb],
}).round(1).head()

## Two more patterns from the paper

These examples are small versions of the two workflows studied in the
[JAZ paper](https://arxiv.org/pdf/2609.26891), where they are applied to much longer tasks.

**Tail-recursive delegation ([Section 4.1](https://arxiv.org/pdf/2609.26891#page=6)).** On a task too long for one context window,
an agent can hand the rest of the task to a subagent *together with its entire history*. Because
`__history__` is passed by reference, nothing is lost (unlike compaction, which summarizes the
history). The subagent receives the history as an input it can search:

```python
return invoke(
    ...,  # the original inputs to the top-level invoke
    prev_history=globals().get("prev_history", []) + __history__,  # lossless, by reference
    next_steps="...",
)
```

This is the "passing context down" direction. On the long-horizon benchmark StuLife, where some
tasks require recalling information from hundreds of tasks earlier, this pattern let an agent search
its full history for those details after delegating dozens of times.

**Self-improvement ([Section 4.2](https://arxiv.org/pdf/2609.26891#page=8)).** A top-level agent can act as a *meta-agent*: it
launches subagents on a sequence of tasks, has each one return its `__history__`, studies the
failures, and improves the inputs (prompts and tools) it gives the next subagents:

```python
answer, trajectory = invoke(task=next_task, instructions=instructions, tool=tool)
```

where `instructions` ends with "When done, return `(answer, __history__)`". This is the "passing
context up" direction, like the example above.

## Scope: inputs for a whole tree of agents

So far, every subagent received exactly what its caller passed it, because
**an input to `invoke` is bound only in that agent's REPL, not in its subagents'**. The top-level
agent had to pass each subagent its chunk, and the notebook.

That's the right default for inputs specific to one subagent, like its chunk. But some inputs
should reach every agent in the tree, such as a tool that every agent may need or an instruction for
all agents. Passing these by hand at every level is tedious, and if one agent forgets, the whole
subtree below it loses the input.

For such inputs, JAZ has `scope`. Values bound with `with scope(name=value):` are inputs to every
`invoke` inside the `with` block, *including all their recursive subagents*.

To see this in action, let's not give the agents `notes` at all. Instead, they can only read the
notes and fill in the table through three tools: `num_notes`, `view_notes` and `set_value`. Every
subagent needs these tools, so we put them in scope, along with a note, `about_tools`, that explains
this to every agent. The agents fill in `table` in place through `set_value`, so this time no agent
returns anything, and all subagents write to the same table.

In [ ]:
from jaz import scope

table = pd.DataFrame(index=notes.index, columns=list(table.columns), dtype=object)  # Filled in by the agents.


def num_notes() -> int:
    """Number of field notes, which is also the number of rows in the table."""
    return len(notes)


def view_notes(start: int, stop: int) -> str:
    """Field notes `start` to `stop - 1`, as text."""
    return notes.iloc[start:stop].to_string()


def set_value(row: int, column: str, value: str | float | None) -> None:
    """Set the cell at (`row`, `column`) of the table to `value`."""
    table.at[row, column] = value


about_tools = (
    "The field notes and the table are not available as DataFrames. Use `num_notes` and "
    "`view_notes` to read the notes, and `set_value` to fill in the table in place."
)
fill_in = (
    "Fill in the table from the field notes, one row per note, with the columns species (Adelie, "
    "Chinstrap or Gentoo), island (Biscoe, Dream or Torgersen), bill_length_mm, bill_depth_mm, "
    "flipper_length_mm, body_mass_g (floats, in mm and g) and sex (MALE or FEMALE). Leave anything "
    "not recorded empty. Make sure to actually read the notes and transcribe them manually; do NOT "
    "use a programmatic parser as it is brittle." + use_subagents
)

with scope(num_notes=num_notes, view_notes=view_notes, set_value=set_value, about_tools=about_tools):
    with RecursionLimit(max_depth=2), Logger(subagents=1, system_prompt=True):
        invoke(task=fill_in)

table

The top-level agent passed each subagent only what was specific to it, yet every subagent could
call `view_notes` and `set_value` and knew that the notes are only reachable through them. Some
details:

- **Scoped inputs are shown in the system prompt.** An explicit input is rendered in the agent's
  user prompt, as in tutorial 1, while a scoped input is rendered in the *system* prompt, followed by
  a note telling the agent that it is also available to every `invoke` it makes. You can see this in
  the log: `root.1`'s user prompt contains only what `root` passed it, while the end of its system
  prompt shows the three tools and `about_tools`. A scoped string like `about_tools` therefore works
  as a system-wide instruction.
- **A scoped name can't also be passed explicitly.** Passing a name that is already in scope as an
  `invoke` keyword argument raises an error, so the two can never disagree.
- **Scoped values are shared, not copied.** Every agent's `set_value` closes over the same `table`,
  which is why all the subagents' edits ended up in one table.

`scope` is the usual way to provide tools: most tools should be available to the whole tree of
agents, not just the top-level one. Hooks and configuration (tutorial 4) follow the same rule:
used as a context manager (`with RecursionLimit(max_depth=2):`), they apply to all `invoke` calls
inside the `with` block, including subagents.

## Summary

- **Subagents are on by default.** Every agent's REPL has `invoke`, so an agent can split a task
  into subtasks and delegate them to subagents, e.g. to process a long input in chunks, RLM-style.
- **Context management through REPL variables.** Everything an agent sees is a variable in its REPL,
  including its own `__history__`, so context can be passed by reference instead of being copied:
  - **up**, from a subagent to its caller, e.g. `return {"rows": rows, "history": __history__}`, so that the caller
    can check the subagent's work, as the meta-agent does in self-improvement;
  - **down**, from an agent to its subagents, e.g. `__history__[0].repl_output`, or a whole
    history in tail-recursive delegation.
- **Inputs don't propagate; `scope` does.** An explicit input reaches only the agent it was passed
  to. A value bound with `with scope(...)` reaches every `invoke` in the block and all their
  subagents, and is rendered in the system prompt. Use it for tools and instructions that every
  agent needs.

---

**Next:** [Hooks, events and effects](03-hooks-events-and-effects.ipynb): what the `Logger` and
`RecursionLimit` hooks you have been using actually are, and how to write your own.